<a href="https://colab.research.google.com/github/yeshaa23/Project-A-Kelompok-4-Pertamina-PBAGenap/blob/main/Notebook/2b_Preprocessing_Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing Khusus BERT (Minimal Cleaning)

## Install dan Import Library

In [3]:
!pip install transformers torch
!pip install tqdm

In [4]:
import pandas as pd
import re
import string
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from tqdm import tqdm
from IPython.display import display

tqdm.pandas()
print("[→] Library siap!")

[→] Library siap!


## Load data hasil scraping

In [9]:
# Load Data Hasil Scraping
from google.colab import drive
# Menghubungkan Google Drive dengan Colab
drive.mount('/content/drive')

import pandas as pd
file_path = '/content/drive/MyDrive/ProjectA-PBA/hasil_scraping_berita.csv'
# Memuat data dari file CSV
df = pd.read_csv(file_path)
# Tampilkan beberapa baris pertama untuk memastikan data dimuat dengan benar
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Netral,money.kompas.com
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positif,money.kompas.com
2,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positif,money.kompas.com
3,https://money.kompas.com/read/2016/04/11/17345...,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",success,Migas,Positif,money.kompas.com
4,https://money.kompas.com/image/2017/11/27/1656...,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,success,Akademik,Positif,money.kompas.com


## Preprocessing sederhana

In [10]:
# ============================================================
# MEMUAT DATA BERITA PERTAMINA & PREPROCESSING DASAR
# ============================================================
print("=" * 60)
print("  PREPROCESSING BERITA PERTAMINA")
print("=" * 60)

# Memuat dataset hasil scraping
df = pd.read_csv(file_path, encoding="utf-8")  # bisa df_raw juga kalau mau tetap
print(f" Data dimuat: {len(df)} baris, {df.shape[1]} kolom")
print(f"\nKolom yang tersedia: {list(df.columns)}")

# ============================================================
# Mapping Sentimen ke English
# ============================================================
sentiment_map = {'Positif':'Positive', 'Negatif':'Negative', 'Netral':'Neutral'}
df['Sentimen'] = (
    df['Sentimen']
    .astype(str)
    .str.strip()
    .str.capitalize()
    .map(sentiment_map)
    .fillna(df['Sentimen'])
)

# ============================================================
# Normalisasi kolom Penerbit
# ============================================================
# Ubah semua penerbit menjadi Title Case + hapus spasi berlebih
df['Penerbit'] = df['Penerbit'].str.title().str.strip()

# Fungsi normalisasi publisher
def normalize_publisher(name):
    if not isinstance(name, str):
        return name

    name = name.strip().lower()  # lowercase + hapus spasi

    # Mapping domain/keyword
    if 'detik' in name:
        return 'Detik'
    elif 'kompas' in name:
        return 'Kompas'
    elif 'antara' in name:
        return 'Antara News'
    elif 'tempo' in name:
        return 'Tempo'
    elif 'kontan' in name:
        return 'Kontan'
    elif 'bisnis' in name:
        return 'Bisnis'
    else:
        return name.title()  # default: title case nama asli

# Terapkan ke kolom Penerbit
df['Penerbit'] = df['Penerbit'].apply(normalize_publisher)

display(df.head(10))

  PREPROCESSING BERITA PERTAMINA
 Data dimuat: 1557 baris, 7 kolom

Kolom yang tersedia: ['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit']


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas
2,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positive,Kompas
3,https://money.kompas.com/read/2016/04/11/17345...,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",success,Migas,Positive,Kompas
4,https://money.kompas.com/image/2017/11/27/1656...,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,success,Akademik,Positive,Kompas
5,https://money.kompas.com/image/2016/11/01/1929...,Foto : Pertamina Tegaskan Kelangkaan BBM Hanya...,"Unduh Kompas.com App untuk berita terkini, aku...",success,BBM,Negative,Kompas
6,https://nasional.kompas.com/read/2017/06/09/20...,Mantan Dirut Pertamina Transkontinental Diteta...,"JAKARTA, KOMPAS.com - Jaksa Agung Muda Pidana ...",success,Hukum,Neutral,Kompas
7,https://otomotif.kompas.com/read/2017/09/16/16...,"Setelah Tol, Beli BBM Juga Wajib Pakai Uang El...","Jakarta, KompasOtomotif - Selain PT Jasa Marga...",success,BBM,Negative,Kompas
8,https://otomotif.kompas.com/read/2017/12/13/11...,Pertamax Turbo buat Konsumen Pemilik Supercar,"Nusa Dua, KompasOtomotif - Pertamina kini teng...",success,Kebakaran,Neutral,Kompas
9,https://money.kompas.com/read/2017/02/11/17000...,PHE WMO Pasok Kondensat ke Industri Thinner da...,"GRESIK, KOMPAS - PT Pertamina Hulu Energi West...",success,Migas,Negative,Kompas


In [20]:
# ============================================================
# Definisi Fungsi Preprocessing Khusus BERT (Minimal Cleaning)
# ============================================================
def preprocess_for_bert(text):
    if not isinstance(text, str):
        return ""

    # 1. Case Folding: semua huruf kecil
    text = text.lower()

    # 2. Menghapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Menghapus Mention (@user) dan Hashtag (#)
    text = re.sub(r'@\w+|#\w+', '', text)

    # 4. Menghapus Angka
    text = re.sub(r'\d+', '', text)

    # 5. Menghapus Tanda Baca
    import string
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 6. Menghapus Whitespace berlebih (spasi ganda, tab, newline)
    text = re.sub(r'\s+', ' ', text).strip()

    # 7. Menghapus Boilerplate Berita yang tidak relevan (case-insensitive)
    boilerplate_patterns = [
        r'kompas\.?com', r'kompascom', r'detik\.com', r'tempo\.co',
        r'tribunnews\.com', r'kontan\.co\.id', r'bisnis\.com',
        r'com', r'co', r'id', r'news', r'finance', r'money', r'otomotif',
        r'jakarta', r'indonesia', r'bojonegoro', r'surabaya',
        r'pt', r'persero', r'tbk'
    ]
    for pattern in boilerplate_patterns:
        text = re.sub(r'\b' + pattern + r'\b', ' ', text, flags=re.IGNORECASE)

    # 8. Bersihkan lagi whitespace setelah boilerplate
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [21]:
# ============================================================
# TERAPKAN KE SETIAP ARTIKEL
# ============================================================
print("[→] Menjalankan preprocessing teks untuk BERT...")
df['content_cleaned_bert'] = df['Isi Berita'].progress_apply(preprocess_for_bert)

# Preview hasil
display(df[['Judul','Isi Berita','content_cleaned_bert']].head())

[→] Menjalankan preprocessing teks untuk BERT...


100%|██████████| 1557/1557 [00:03<00:00, 393.86it/s]


,Judul,Isi Berita,content_cleaned_bert
0,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,diskusi mengenai csr di industri pelumas berta...
1,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",anak perusahaan pertamina yakni pertamina lubr...
2,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,pelepasan indukan tuntong laut oleh tim konser...
3,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",pemerintah terus mengupayakan penambahan produ...
4,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,direktur operasional pertamina patra niaga abd...


## Simpan hasil preprocessing

In [22]:
# ============================================================
# SIMPAN DATA FINAL
# ============================================================
OUTPUT_FILE = "hasil_preprocessing_bert.csv"
OUTPUT_PATH = f"/content/drive/MyDrive/ProjectA-PBA/{OUTPUT_FILE}"

# Kolom final + kolom asli
columns_to_save = ['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit', 'content_cleaned_bert']

df[columns_to_save].to_csv(OUTPUT_PATH, index=False)
print(f"Dataset final berhasil disimpan: {OUTPUT_PATH}")

Dataset final berhasil disimpan: /content/drive/MyDrive/ProjectA-PBA/hasil_preprocessing_bert.csv
